In [0]:
%sql
CREATE TABLE deltasource
(
  id INT,
  name STRING,
  salary INT 
)
USING DELTA 
LOCATION '/FileStore/deltasource/source1'

In [0]:
%sql
ALTER TABLE deltasource SET TBLPROPERTIES ('delta.enableDeletionVectors' = false)

In [0]:
%sql
INSERT INTO deltasource
VALUES 
(1, "priya", 100),
(2, "rahul", 150),
(3, "shanti", 200)

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
UPDATE deltasource
SET name = "iron" where id = 2;

num_affected_rows
7


In [0]:
%sql
INSERT INTO deltasource
VALUES 
(1, "priya", 100),
(2, "rahul", 150),
(3, "shanti", 200)

num_affected_rows,num_inserted_rows
3,3


In [0]:
df = spark.readStream.option("startingVersion",8).table("deltasource")

In [0]:
df_new = spark.readStream.option("startingVersion",8).option("ignoreChanges",True).table("deltasource")

df_new.writeStream.format("delta")\
        .option("checkpointLocation","/FileStore/deltasource/sink1/checkpoint_new")\
        .option("path","/FileStore/deltasource/sink1/data_new")\
        .trigger(processingTime = "3 seconds")\
        .start()

Out[29]: <pyspark.sql.streaming.query.StreamingQuery at 0x7f6475222190>

In [0]:
df.writeStream.format("delta")\
        .option("checkpointLocation","/FileStore/deltasource/sink1/checkpoint")\
        .option("path","/FileStore/deltasource/sink1/data")\
        .trigger(processingTime = "3 seconds")\
        .start()

Out[23]: <pyspark.sql.streaming.query.StreamingQuery at 0x7f64752711c0>

In [0]:
%sql
DESCRIBE HISTORY deltasource

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2025-02-19T00:01:32.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
7,2025-02-19T00:01:28.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
6,2025-02-19T00:01:22.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
5,2025-02-18T23:57:54.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
4,2025-02-18T23:57:34.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
3,2025-02-18T23:55:26.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
2,2025-02-18T23:46:10.000+0000,6806717008824971,anshlamba8@gmail.com,WRITE,"Map(mode -> Append, partitionBy -> [])",null,List(892053813456100),0218-232157-kyvmc7t7,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1068)",null,Databricks-Runtime/12.2.x-scala2.12
1,2025-02-18T23:44:03.000+0000,6806717008824971,anshlamba8@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableDeletionVectors"":""false""})",null,List(892053813456100),0218-232157-kyvmc7t7,0,WriteSerializable,true,Map(),null,Databricks-Runtime/12.2.x-scala2.12
0,2025-02-18T23:41:55.000+0000,6806717008824971,anshlamba8@gmail.com,CREATE TABLE,"Map(isManaged -> false, description -> null, partitionBy -> [], properties -> {})",null,List(892053813456100),0218-232157-kyvmc7t7,null,WriteSerializable,true,Map(),null,Databricks-Runtime/12.2.x-scala2.12


In [0]:
%sql
SELECT * FROM delta.`/FileStore/deltasource/sink1/data_new`

id,name,salary
1,priya,100
2,rahul,150
3,shanti,200
1,iron,100
2,rahul,150
3,shanti,200
1,iron,100
2,rahul,150
3,shanti,200
1,iron,100
